In [0]:
%sql

CREATE CATALOG IF NOT EXISTS orders_demo
MANAGED LOCATION 'abfss://adlscontainer1@adls2501.dfs.core.windows.net/catalogs/orders_demo';



In [0]:
%sql
CREATE SCHEMA  IF NOT EXISTS orders_demo.bronze;

In [0]:
%sql
CREATE SCHEMA  IF NOT EXISTS orders_demo.silver;

In [0]:
%python
    "abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv",
    header=True,
    inferSchema=True
)
display(df)

empno,ename,sal
e1,babjee,100000
e2,naveen,200000
e3,sunil,40000
e4,sruthi,40000
e5,sakshi,30000


In [0]:
%python
df.write.mode("overwrite").parquet("abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.parquet")


In [0]:

import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

// File location
val rawPath =
  "abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv"

val bronzeTable = "orders_demo.bronze.employees_raw"

// Read CSV using comma separator
val dfRaw = spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("sep", ",")
  .load(rawPath)

// Check column names BEFORE writing
println(dfRaw.columns.mkString(", "))

display(dfRaw)


empno,ename,sal
e1,babjee,100000
e2,naveen,200000
e3,sunil,40000
e4,sruthi,40000
e5,sakshi,30000


In [0]:

val dfBronze = dfRaw
  .withColumn("ingest_ts", current_timestamp())
  .withColumn("source_file", input_file_name())

dfBronze.write
  .format("delta")
  .mode("append")
  .option("mergeSchema", "true")
  .saveAsTable("orders_demo.bronze.employees_raw")

display(spark.table("orders_demo.bronze.employees_raw"))


empno,ename,sal,ingest_ts,source_file
e1,babjee,100000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e2,naveen,200000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e3,sunil,40000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e4,sruthi,40000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e5,sakshi,30000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv


Silver: Clean and transform


In [0]:

import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

val bronzeTable = "orders_demo.bronze.employees_raw"
val silverTable = "orders_demo.silver.employees"

// Read Bronze table
val dfClean = spark.table(bronzeTable)

  // Remove records without employee ID
  .filter(col("empno").isNotNull)

  // Remove records without employee name
  .filter(col("ename").isNotNull)

  // Remove empty employee names
  .filter(trim(col("ename")) =!= "")

  // Clean employee ID and name
  .withColumn("empno", trim(col("empno")))
  .withColumn("ename", trim(col("ename")))

  // Convert salary to decimal
  .withColumn("sal", col("sal").cast(DecimalType(12, 2)))

  // Keep only positive salaries
  .filter(col("sal") > 0)

  // Remove duplicate employees
  .dropDuplicates(Seq("empno"))

// Register temporary view
dfClean.createOrReplaceTempView("employees_clean")

// Display cleaned data
display(dfClean)


empno,ename,sal,ingest_ts,source_file
e1,babjee,100000.00,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e2,naveen,200000.00,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e3,sunil,40000.00,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e4,sruthi,40000.00,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv
e5,sakshi,30000.00,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv


Silver Delta table

In [0]:

val silverTable = "orders_demo.silver.employees"

spark.sql(s"""
  CREATE TABLE IF NOT EXISTS $silverTable (
    empno STRING,
    ename STRING,
    sal DECIMAL(12,2),
    updated_at TIMESTAMP
  )
  USING DELTA
""")

println("Silver employees table created successfully")


Silver employees table created successfully
val silverTable: String = orders_demo.silver.employees

MERGE

In [0]:

spark.sql(s"""
  MERGE INTO $silverTable AS t
  USING employees_clean AS s
  ON t.empno = s.empno

  WHEN MATCHED THEN UPDATE SET
    t.ename = s.ename,
    t.sal = s.sal,
    t.updated_at = current_timestamp()

  WHEN NOT MATCHED THEN INSERT (
    empno,
    ename,
    sal,
    updated_at
  )
  VALUES (
    s.empno,
    s.ename,
    s.sal,
    current_timestamp()
  )
""")

println("Employee MERGE completed successfully")


Employee MERGE completed successfully

Verify Silver table

In [0]:

display(spark.table("orders_demo.silver.employees"))


empno,ename,sal,updated_at
e1,babjee,100000.00,2026-11-16T09:42:58.749042Z
e2,naveen,200000.00,2026-11-16T09:42:58.749042Z
e3,sunil,40000.00,2026-11-16T09:42:58.749042Z
e4,sruthi,40000.00,2026-11-16T09:42:58.749042Z
e5,sakshi,30000.00,2026-11-16T09:42:58.749042Z


In [0]:

val employeeUpdates = Seq(
  ("e2", "naveen", BigDecimal(250000))
).toDF("empno", "ename", "sal")

employeeUpdates.createOrReplaceTempView("employee_updates")

spark.sql(s"""
  MERGE INTO $silverTable AS t
  USING employee_updates AS s
  ON t.empno = s.empno

  WHEN MATCHED THEN UPDATE SET
    t.ename = s.ename,
    t.sal = s.sal,
    t.updated_at = current_timestamp()

  WHEN NOT MATCHED THEN INSERT (
    empno, ename, sal, updated_at
  )
  VALUES (
    s.empno, s.ename, s.sal, current_timestamp()
  )
""")

display(spark.table(silverTable))


empno,ename,sal,updated_at
e2,naveen,250000.00,2026-11-16T09:44:05.778819Z
e1,babjee,100000.00,2026-11-16T09:42:58.749042Z
e3,sunil,40000.00,2026-11-16T09:42:58.749042Z
e4,sruthi,40000.00,2026-11-16T09:42:58.749042Z
e5,sakshi,30000.00,2026-11-16T09:42:58.749042Z


Delta Lake Time Travel

In [0]:

%sql
DESCRIBE HISTORY orders_demo.silver.employees;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-16T09:44:12Z,141528520913855,shobika2501@gmail.com,MERGE,"Map(predicate -> [""(empno#1279 = empno#1272)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3363117735064514),null,1116-082706-5jzydfa3,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2570, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6116, materializeSourceTimeMs -> 174, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1562, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4349)",null,Databricks-Runtime/19.x-cpu-ml-scala2.13
1,2026-11-16T09:43:03Z,141528520913855,shobika2501@gmail.com,MERGE,"Map(predicate -> [""(empno#641 = empno#545)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3363117735064514),null,1116-082706-5jzydfa3,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1847, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4876, materializeSourceTimeMs -> 909, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3191, numTargetRowsUpdated -> 0, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 718)",null,Databricks-Runtime/19.x-cpu-ml-scala2.13
0,2026-11-16T09:42:27Z,141528520913855,shobika2501@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b383860a-aafa-466d-903c-62e9ea406aac"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-b09de967-e2cc-4b0d-9429-f80799c48bd5""}, statsOnLoad -> false)",null,List(3363117735064514),null,1116-082706-5jzydfa3,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.x-cpu-ml-scala2.13


In [0]:

val history = spark.sql(
  "DESCRIBE HISTORY orders_demo.silver.employees"
)

display(history)


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-16T09:44:12Z,141528520913855,shobika2501@gmail.com,MERGE,"Map(predicate -> [""(empno#1279 = empno#1272)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3363117735064514),null,1116-082706-5jzydfa3,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2570, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6116, materializeSourceTimeMs -> 174, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1562, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4349)",null,Databricks-Runtime/19.x-cpu-ml-scala2.13
1,2026-11-16T09:43:03Z,141528520913855,shobika2501@gmail.com,MERGE,"Map(predicate -> [""(empno#641 = empno#545)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3363117735064514),null,1116-082706-5jzydfa3,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1847, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4876, materializeSourceTimeMs -> 909, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3191, numTargetRowsUpdated -> 0, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 718)",null,Databricks-Runtime/19.x-cpu-ml-scala2.13
0,2026-11-16T09:42:27Z,141528520913855,shobika2501@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b383860a-aafa-466d-903c-62e9ea406aac"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-b09de967-e2cc-4b0d-9429-f80799c48bd5""}, statsOnLoad -> false)",null,List(3363117735064514),null,1116-082706-5jzydfa3,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.x-cpu-ml-scala2.13


In [0]:

val oldData = spark.read
  .format("delta")
  .option("versionAsOf", 0)
  .table("orders_demo.silver.employees")

display(oldData)


empno,ename,sal,updated_at


In [0]:

val latestData = spark.table("orders_demo.silver.employees")

display(latestData)


empno,ename,sal,updated_at
e2,naveen,250000.00,2026-11-16T09:44:05.778819Z
e1,babjee,100000.00,2026-11-16T09:42:58.749042Z
e3,sunil,40000.00,2026-11-16T09:42:58.749042Z
e4,sruthi,40000.00,2026-11-16T09:42:58.749042Z
e5,sakshi,30000.00,2026-11-16T09:42:58.749042Z


Schema Evolution

In [0]:

import spark.implicits._
import org.apache.spark.sql.functions._

// New employee records with department
val newEmployees = Seq(
  ("e1", "babjee", 100000.0, "IT"),
  ("e2", "naveen", 250000.0, "HR"),
  ("e6", "arun", 60000.0, "Finance")
).toDF("empno", "ename", "sal", "department")

// Add ingestion timestamp
val newBronzeData = newEmployees
  .withColumn("sal", col("sal").cast("decimal(12,2)"))
  .withColumn("ingest_ts", current_timestamp())

// Append with schema evolution
newBronzeData.write
  .format("delta")
  .mode("append")
  .option("mergeSchema", "true")
  .saveAsTable("orders_demo.bronze.employees_raw")

println("New employee data appended successfully")

display(spark.table("orders_demo.bronze.employees_raw"))


empno,ename,sal,ingest_ts,source_file,department
e1,babjee,100000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv,null
e2,naveen,200000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv,null
e3,sunil,40000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv,null
e4,sruthi,40000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv,null
e5,sakshi,30000,2026-11-16T09:39:40.225742Z,abfss://adlscontainer1@adls2501.dfs.core.windows.net/emp.csv,null
e1,babjee,100000,2026-11-16T09:47:37.998902Z,null,IT
e2,naveen,250000,2026-11-16T09:47:37.998902Z,null,HR
e6,arun,60000,2026-11-16T09:47:37.998902Z,null,Finance


In [0]:

%sql
ALTER TABLE orders_demo.silver.employees
ADD COLUMN department STRING;


In [0]:

val employeeUpdates = newEmployees
  .withColumn("sal", col("sal").cast("decimal(12,2)"))

employeeUpdates.createOrReplaceTempView("employee_updates")


val employeeUpdates: org.apache.spark.sql.DataFrame = [empno: string, ename: string ... 2 more fields]

In [0]:

spark.sql("""
  MERGE INTO orders_demo.silver.employees AS t
  USING employee_updates AS s
  ON t.empno = s.empno

  WHEN MATCHED THEN UPDATE SET
    t.ename = s.ename,
    t.sal = s.sal,
    t.department = s.department,
    t.updated_at = current_timestamp()

  WHEN NOT MATCHED THEN INSERT (
    empno, ename, sal, department, updated_at
  )
  VALUES (
    s.empno, s.ename, s.sal, s.department, current_timestamp()
  )
""")

display(spark.table("orders_demo.silver.employees"))


empno,ename,sal,updated_at,department
e1,babjee,100000.00,2026-11-16T09:48:32.684472Z,IT
e2,naveen,250000.00,2026-11-16T09:48:32.684472Z,HR
e6,arun,60000.00,2026-11-16T09:48:32.684472Z,Finance
e3,sunil,40000.00,2026-11-16T09:42:58.749042Z,null
e4,sruthi,40000.00,2026-11-16T09:42:58.749042Z,null
e5,sakshi,30000.00,2026-11-16T09:42:58.749042Z,null


In [0]:

%sql
OPTIMIZE orders_demo.silver.employees;


path,metrics
abfss://adlscontainer1@adls2501.dfs.core.windows.net/catalogs/orders_demo/__unitystorage/catalogs/65e5f3df-efbb-4e63-8bad-75c9096e0ba4/tables/ac57883d-9d46-4511-ba04-b1e2802272c5,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1794822549487, 1794822553601, 4, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null, 0)"


In [0]:

%sql
OPTIMIZE orders_demo.silver.employees
ZORDER BY (empno);


path,metrics
abfss://adlscontainer1@adls2501.dfs.core.windows.net/catalogs/orders_demo/__unitystorage/catalogs/65e5f3df-efbb-4e63-8bad-75c9096e0ba4/tables/ac57883d-9d46-4511-ba04-b1e2802272c5,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3081), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1794822579014, 1794822581170, 4, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null, 0)"


In [0]:

%sql
VACUUM orders_demo.silver.employees DRY RUN;


path


In [0]:

%sql
VACUUM orders_demo.silver.employees RETAIN 168 HOURS;


path
abfss://adlscontainer1@adls2501.dfs.core.windows.net/catalogs/orders_demo/__unitystorage/catalogs/65e5f3df-efbb-4e63-8bad-75c9096e0ba4/tables/ac57883d-9d46-4511-ba04-b1e2802272c5
